In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
query = """
SELECT
    order_item.id AS "Order Item ID",
    order_item.quantity AS "Quantity",
    product.id AS "Product ID",
    product.name AS "Product Name",
    product.price AS "Product Price",
    restaurant_order.id AS "Order ID",
    restaurant_order.datetime AS "Order Datetime"
FROM restaurant_orderitem AS order_item
JOIN restaurant_product AS product
    ON product.id = order_item.product_id
JOIN restaurant_order
    ON restaurant_order.id = order_item.order_id
"""

with sqlite3.connect("../db.sqlite3") as connection:
    joined_data = pd.read_sql_query(query, connection)

csv_path = "../restaurant_data.csv"
joined_data.to_csv(csv_path, index=False)
data = pd.read_csv(csv_path, parse_dates=["Order Datetime"])

print(f"Dataframe shape: {data.shape}")
data.head()

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_products_by_quantity = (
    data.groupby("Product Name")["Quantity"].sum().nlargest(10)
)

print(top_products_by_quantity)
top_products_by_quantity.plot.pie(
    autopct="%1.1f%%",
    figsize=(10, 8),
    startangle=90,
    ylabel="",
    title="Top 10 products by quantity sold",
)
plt.tight_layout()
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
data["Item Price"] = data["Product Price"].mul(data["Quantity"])
top_products_by_income = (
    data.groupby("Product Name")["Item Price"].sum().nlargest(10)
)

print(top_products_by_income)
top_products_by_income.plot.pie(
    autopct="%1.1f%%",
    figsize=(10, 8),
    startangle=90,
    ylabel="",
    title="Top 10 products by restaurant income",
)
plt.tight_layout()
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
data["Order Hour"] = data["Order Datetime"].dt.hour
income_by_hour = (
    data.groupby("Order Hour")["Item Price"]
    .sum()
    .reindex(np.arange(24), fill_value=0)
)

print(income_by_hour)
income_by_hour.plot.bar(
    figsize=(12, 6),
    color="steelblue",
    rot=0,
    xlabel="Order hour",
    ylabel="Total income",
    title="Restaurant income by order hour",
)
plt.tight_layout()
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
weekday_order = pd.Index(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
)
data["Order Day Of The Week"] = data["Order Datetime"].dt.day_name()
income_by_weekday = (
    data.groupby("Order Day Of The Week")["Item Price"]
    .sum()
    .reindex(weekday_order)
)

print(income_by_weekday)
income_by_weekday.plot.bar(
    figsize=(10, 6),
    color="steelblue",
    rot=30,
    xlabel="Day of the week",
    ylabel="Total income",
    title="Restaurant income by day of the week",
)
plt.tight_layout()
plt.show()